# Gradient Boosting sobre el dataset Diabetes 130-US Hospitals

Este notebook aplica modelos de boosting al dataset `diabetes_preprocesado.csv`, centrándose en **AdaBoost** y **XGBoost**.

La prioridad de esta versión es que los resultados sean **comparables** con los notebooks de árbol de decisión y random forest del grupo. Por tanto, se mantiene el mismo protocolo común:

- Se usa `readmitted` como variable objetivo sin convertirla a binaria.
- Se usa `X = df.drop(columns=["readmitted"])`.
- Se usa `train_test_split` con `test_size=0.2`, `random_state=42` y `stratify=y`.
- Se evalúa con `accuracy`, `precision`, `recall` y `F1-score`.

Como `readmitted` tiene más de dos clases, las métricas de precision, recall y F1 se calculan con media **macro**, igual que en el notebook de árbol de decisión.

## 1. Importación de librerías

Se importan las librerías necesarias para cargar los datos, dividir el conjunto en entrenamiento y prueba, entrenar los modelos y calcular las métricas solicitadas.

In [2]:
# Instalación de dependencias necesarias para este notebook
# Ejecutar esta celda solo si alguna librería no está instalada en el entorno.

%pip install -q pandas numpy scikit-learn matplotlib seaborn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        "No se ha encontrado la librería xgboost. Instálala con: pip install xgboost"
    ) from exc

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Carga del dataset preprocesado

El notebook busca el archivo en varias rutas habituales para que pueda ejecutarse tanto desde la raíz del repositorio como desde una carpeta de notebooks.

In [ ]:
DATA_PATH = 

if DATA_PATH is None:
    raise FileNotFoundError(
        "No se ha encontrado diabetes_preprocesado.csv. "
        "Coloca el archivo junto al notebook, en data/, en ../data/ o actualiza DATA_PATH."
    )

df = pd.read_csv(DATA_PATH, sep=",", quotechar='"')

print(f"Dataset cargado desde: {DATA_PATH}")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

df.head()

FileNotFoundError: No se ha encontrado diabetes_preprocesado.csv. Coloca el archivo junto al notebook, en data/, en ../data/ o actualiza DATA_PATH.

## 3. Comprobación de la variable objetivo

Para que la comparación sea justa con los demás modelos del grupo, se usa directamente la columna `readmitted` tal y como aparece en el CSV preprocesado.

No se transforma a problema binario en este notebook, porque los notebooks de árbol de decisión y random forest trabajan con la variable objetivo original.

In [ ]:
if "readmitted" not in df.columns:
    raise ValueError("No se encuentra la columna objetivo 'readmitted' en el dataset.")

print("Distribución de la variable objetivo readmitted:")
target_counts = df["readmitted"].value_counts().sort_index()
target_percentages = df["readmitted"].value_counts(normalize=True).sort_index() * 100

target_distribution = pd.DataFrame({
    "conteo": target_counts,
    "porcentaje": target_percentages.round(2)
})

display(target_distribution)

## 4. Separación en variables predictoras y variable objetivo

Se replica el criterio usado en los otros notebooks:

```python
y = df["readmitted"]
X = df.drop(columns=["readmitted"])
```

Esto garantiza que AdaBoost y XGBoost se evalúen sobre las mismas columnas de entrada que los modelos de árbol de decisión y random forest.

In [ ]:
y = df["readmitted"]
X = df.drop(columns=["readmitted"])

print(f"Dimensiones de X: {X.shape}")
print(f"Dimensiones de y: {y.shape}")
print(f"Número de clases: {y.nunique()}")
print(f"Clases: {sorted(y.unique())}")

## 5. División en entrenamiento y prueba

Se utiliza la misma división que en los notebooks del grupo:

- 80 % entrenamiento.
- 20 % prueba.
- `random_state=42`.
- `stratify=y` para conservar la proporción de clases.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train: {X_train.shape[0]:,} muestras")
print(f"Test: {X_test.shape[0]:,} muestras")

split_distribution = pd.DataFrame({
    "train_%": (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
    "test_%": (y_test.value_counts(normalize=True).sort_index() * 100).round(2)
})

display(split_distribution)

## 6. Funciones auxiliares de evaluación

Como la tarea es multiclase, se calculan `precision`, `recall` y `F1-score` con media **macro**. Esta media da el mismo peso a cada clase, algo importante cuando existe desbalanceo.

La métrica principal para comparar los modelos será `F1 Macro`, ya que combina precision y recall y no queda dominada por la clase mayoritaria.

In [ ]:
def calculate_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision Macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall Macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 Macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def train_and_evaluate(model_name, model, X_train, y_train, X_test, y_test):
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    y_pred = model.predict(X_test)
    metrics = calculate_metrics(y_test, y_pred)

    row = {
        "Modelo": model_name,
        **metrics,
        "Tiempo entrenamiento (s)": training_time
    }

    return model, y_pred, row


def format_results(results):
    results_df = pd.DataFrame(results)
    metric_columns = ["Accuracy", "Precision Macro", "Recall Macro", "F1 Macro", "Tiempo entrenamiento (s)"]
    results_df[metric_columns] = results_df[metric_columns].round(4)
    return results_df.sort_values("F1 Macro", ascending=False).reset_index(drop=True)

## 7. Modelo de referencia simple

Antes de entrenar AdaBoost y XGBoost, se incluye un clasificador trivial que predice siempre la clase más frecuente. No es el modelo que se pretende usar, pero sirve para comprobar si los modelos reales superan una referencia mínima.

In [ ]:
all_results = {}
all_predictions = {}

dummy_model = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)

dummy_model, dummy_pred, dummy_row = train_and_evaluate(
    "Dummy - clase mayoritaria",
    dummy_model,
    X_train,
    y_train,
    X_test,
    y_test
)

all_results["Dummy - clase mayoritaria"] = dummy_row
all_predictions["Dummy - clase mayoritaria"] = dummy_pred

display(format_results([dummy_row]))

## 8. AdaBoost

AdaBoost combina muchos clasificadores débiles, normalmente árboles muy simples, entrenándolos de forma secuencial. En cada iteración presta más atención a las muestras que los modelos anteriores clasificaron mal.

En esta sección se comparan varias configuraciones sencillas de AdaBoost modificando el número de estimadores y el `learning_rate`.

In [ ]:
adaboost_configs = [
    {
        "name": "AdaBoost - 25 estimadores, lr=1.0",
        "params": {
            "estimator": DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            "n_estimators": 25,
            "learning_rate": 1.0,
            "random_state": RANDOM_STATE
        }
    },
    {
        "name": "AdaBoost - 50 estimadores, lr=1.0",
        "params": {
            "estimator": DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            "n_estimators": 50,
            "learning_rate": 1.0,
            "random_state": RANDOM_STATE
        }
    },
    {
        "name": "AdaBoost - 75 estimadores, lr=0.5",
        "params": {
            "estimator": DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            "n_estimators": 75,
            "learning_rate": 0.5,
            "random_state": RANDOM_STATE
        }
    }
]

adaboost_results = []

for config in adaboost_configs:
    model = AdaBoostClassifier(**config["params"])

    trained_model, y_pred, row = train_and_evaluate(
        config["name"],
        model,
        X_train,
        y_train,
        X_test,
        y_test
    )

    adaboost_results.append(row)
    all_results[config["name"]] = row
    all_predictions[config["name"]] = y_pred

adaboost_results_df = format_results(adaboost_results)
display(adaboost_results_df)

## 9. Mejor modelo AdaBoost

Se selecciona como mejor configuración de AdaBoost aquella que obtiene mayor `F1 Macro` en el conjunto de prueba.

In [ ]:
best_adaboost_name = adaboost_results_df.iloc[0]["Modelo"]
print(f"Mejor configuración de AdaBoost según F1 Macro: {best_adaboost_name}")

best_adaboost_metrics = pd.DataFrame([all_results[best_adaboost_name]]).round(4)
display(best_adaboost_metrics)

## 10. XGBoost

XGBoost es una implementación eficiente de gradient boosting basada en árboles de decisión. A diferencia de AdaBoost, optimiza una función de pérdida mediante boosting gradiente e incorpora regularización y técnicas de optimización que suelen mejorar el rendimiento en datos tabulares.

Se evalúan varias configuraciones ligeras para mantener el notebook reproducible y razonablemente rápido.

In [ ]:
num_classes = y.nunique()

xgboost_configs = [
    {
        "name": "XGBoost - 50 est., depth=3, lr=0.10",
        "params": {
            "objective": "multi:softprob",
            "num_class": num_classes,
            "eval_metric": "mlogloss",
            "n_estimators": 50,
            "max_depth": 3,
            "learning_rate": 0.10,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "random_state": RANDOM_STATE,
            "n_jobs": 2,
            "tree_method": "hist"
        }
    },
    {
        "name": "XGBoost - 100 est., depth=3, lr=0.05",
        "params": {
            "objective": "multi:softprob",
            "num_class": num_classes,
            "eval_metric": "mlogloss",
            "n_estimators": 100,
            "max_depth": 3,
            "learning_rate": 0.05,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "random_state": RANDOM_STATE,
            "n_jobs": 2,
            "tree_method": "hist"
        }
    },
    {
        "name": "XGBoost - 100 est., depth=4, lr=0.05",
        "params": {
            "objective": "multi:softprob",
            "num_class": num_classes,
            "eval_metric": "mlogloss",
            "n_estimators": 100,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "random_state": RANDOM_STATE,
            "n_jobs": 2,
            "tree_method": "hist"
        }
    }
]

xgboost_results = []

for config in xgboost_configs:
    model = XGBClassifier(**config["params"])

    trained_model, y_pred, row = train_and_evaluate(
        config["name"],
        model,
        X_train,
        y_train,
        X_test,
        y_test
    )

    xgboost_results.append(row)
    all_results[config["name"]] = row
    all_predictions[config["name"]] = y_pred

xgboost_results_df = format_results(xgboost_results)
display(xgboost_results_df)

## 11. Mejor modelo XGBoost

Se selecciona como mejor configuración de XGBoost aquella que obtiene mayor `F1 Macro` en el conjunto de prueba.

In [ ]:
best_xgboost_name = xgboost_results_df.iloc[0]["Modelo"]
print(f"Mejor configuración de XGBoost según F1 Macro: {best_xgboost_name}")

best_xgboost_metrics = pd.DataFrame([all_results[best_xgboost_name]]).round(4)
display(best_xgboost_metrics)

## 12. Comparación final de modelos

En esta tabla se comparan todos los modelos entrenados. Para elegir el mejor modelo dentro de este notebook se usa `F1 Macro`, porque permite evaluar el equilibrio entre precision y recall en una tarea multiclase con clases desbalanceadas.

In [ ]:
final_results_df = format_results(list(all_results.values()))
display(final_results_df)

## 13. Informe detallado del mejor modelo global

Se identifica el modelo con mejor `F1 Macro` y se muestra un informe de clasificación con las métricas por clase.

In [ ]:
best_global_name = final_results_df.iloc[0]["Modelo"]
best_global_pred = all_predictions[best_global_name]

print(f"Mejor modelo global según F1 Macro: {best_global_name}")

print("\nClassification report:")
print(classification_report(y_test, best_global_pred, zero_division=0))

## 14. Matriz de confusión

La matriz de confusión permite ver qué clases confunde más el mejor modelo. Esto es útil porque una buena accuracy global puede ocultar errores importantes en clases minoritarias.

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, best_global_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real {label}" for label in labels],
    columns=[f"Predicho {label}" for label in labels]
)

display(cm_df)

## 15. Importancia de variables en XGBoost

Cuando el mejor modelo de XGBoost está entrenado, se puede consultar la importancia de variables. Esta información no debe interpretarse como causalidad, pero ayuda a identificar qué variables han sido más utilizadas por el modelo.

In [ ]:
best_xgb_config = next(config for config in xgboost_configs if config["name"] == best_xgboost_name)

best_xgb_model = XGBClassifier(**best_xgb_config["params"])
best_xgb_model.fit(X_train, y_train)

feature_importances = pd.DataFrame({
    "feature": X.columns,
    "importance": best_xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importances.head(20))

## 16. Conclusión

Este notebook evalúa AdaBoost y XGBoost siguiendo el mismo protocolo de partición y variable objetivo que los notebooks del grupo. Por tanto, sus resultados pueden compararse con los obtenidos por árbol de decisión y random forest.

La comparación final debe hacerse usando la misma métrica acordada por el grupo. En este notebook se recomienda usar `F1 Macro`, ya que el problema es multiclase y existe desbalanceo entre las clases.